# Final Project: Parsed and Annotated Data + Derived Tables

### Set Up

In [56]:
import pandas as pd
import numpy as np
from glob import glob
import re
import nltk
import plotly_express as px
from numpy.linalg import norm

In [57]:
import configparser

In [58]:
config = configparser.ConfigParser()
config.read("../../env.ini")
data_home = config['DEFAULT']['data_home']
output_dir = config['DEFAULT']['output_dir']
local_lib = config['DEFAULT']['local_lib']

In [59]:
data_prefix = 'dostoevsky'
source_files = f'{data_home}/gutenberg/dostoevsky-set'

In [60]:
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']

In [61]:
import sys
sys.path.append(local_lib)

In [62]:
from textparser import TextParser

# Parsed and Annotated Data

## Creating LIB Table

In [63]:
clip_pats = [
    r"\*\*\*\s*START OF",
    r"\*\*\*\s*END OF"
]

ohco_pat_list = [
    (2554,   rf"^(CHAPTER|EPILOGUE)"),
    (2638,   rf"^(I|II|III|IV|V|VI|VII|VIII|IX|X|XI|XII|XIII|XIV|XV|XVI)\.$"),
    (28054,  rf"^Chapter [IVX]+\.$"),
    (600,  rf"^(I|II|III|IV|V|VI|VII|VIII|IX|X|XI)$"),
    (8117,  rf"^CHAPTER.*"),
    (2197,  rf"^(I|II|III|IV|V|VI|VII|VIII|IX|X|XI|XII|XIII|XIV|XV|XVI|XVII)$")
]


In [64]:
source_file_list = sorted(glob(f"{source_files}/*.*"))

In [65]:
book_data = []
for source_file_path in source_file_list:
    book_id = int(source_file_path.split('-')[-1].split('.')[0].replace('pg',''))
    book_title = source_file_path.split('/')[-1].split('-')[0].replace('_', ' ')
    book_data.append((book_id, source_file_path, book_title))

In [66]:
LIB = pd.DataFrame(book_data, columns=['book_id','source_file_path','raw_title'])\
    .set_index('book_id').sort_index()

In [67]:
 try:
     LIB['author'] = LIB.raw_title.apply(lambda x: ', '.join(x.split()[:2]))
     LIB['title'] = LIB.raw_title.apply(lambda x: ' '.join(x.split()[2:]))
     LIB = LIB.drop('raw_title', axis=1)
 except AttributeError:
     pass

### Setting up the three features used for model summarization:

In [68]:
# publication dates
dates = {
    600: 1864,
    2197: 1866,
    2554: 1866,
    2638: 1869,
    8117: 1872,
    28054: 1880
}
LIB['date'] = LIB.index.map(dates)

In [69]:
# document types
types = {
    600: 'novella',
    2197: 'novella',
    2554: 'novel',
    2638: 'novel',
    8117: 'novel',
    28054: 'novel'
}
LIB['type'] = LIB.index.map(types)

**Sources differ on the exact classification of *The Gambler,* however most classify it as either "novella" or "short novel." Additionally, it's book_len and n_chaps (seen below) values are more similar to agreed-upon-novella *Notes From the Underground,* so it will be classified as a novella in this project.**

In [70]:
# translators
translators = {
    600: 'Constance Garnett',
    2197: 'C. J. Hogarth',
    2554: 'Constance Garnett',
    2638: 'Eva Martin',
    8117: 'Constance Garnett',
    28054: 'Constance Garnett'
}
LIB['translator'] = LIB.index.map(translators)

In [71]:
LIB['chap_regex'] = LIB.index.map(pd.Series({x[0]:x[1] for x in ohco_pat_list}))

In [72]:
LIB

,source_file_path,author,title,date,type,translator,chap_regex
book_id,,,,,,,
600,/home/wgp3aq/Documents/MSDS/DS5001/data/gutenb...,"DOSTOEVSKY, FYODOR",NOTES FROM THE UNDERGROUND,1864,novella,Constance Garnett,^(I|II|III|IV|V|VI|VII|VIII|IX|X|XI)$
2197,/home/wgp3aq/Documents/MSDS/DS5001/data/gutenb...,"DOSTOEVSKY, FYODOR",THE GAMBLER,1866,novella,C. J. Hogarth,^(I|II|III|IV|V|VI|VII|VIII|IX|X|XI|XII|XIII|X...
2554,/home/wgp3aq/Documents/MSDS/DS5001/data/gutenb...,"DOSTOEVSKY, FYODOR",CRIME AND PUNISHMENT,1866,novel,Constance Garnett,^(CHAPTER|EPILOGUE)
2638,/home/wgp3aq/Documents/MSDS/DS5001/data/gutenb...,"DOSTOEVSKY, FYODOR",THE IDIOT,1869,novel,Eva Martin,^(I|II|III|IV|V|VI|VII|VIII|IX|X|XI|XII|XIII|X...
8117,/home/wgp3aq/Documents/MSDS/DS5001/data/gutenb...,"DOSTOEVSKY, FYODOR",THE POSSESSED,1872,novel,Constance Garnett,^CHAPTER.*
28054,/home/wgp3aq/Documents/MSDS/DS5001/data/gutenb...,"DOSTOEVSKY, FYODOR",THE BROTHERS KARAMAZOV,1880,novel,Constance Garnett,^Chapter [IVX]+\.$


## Creating CORPUS Table

In [73]:
def tokenize_collection(LIB):

    clip_pats = [
        r"\*\*\*\s*START OF",
        r"\*\*\*\s*END OF"
    ]

    books = []
    for book_id in LIB.index:

        # Announce
        print("Tokenizing", book_id, LIB.loc[book_id].title)

        # Define vars
        chap_regex = LIB.loc[book_id].chap_regex
        ohco_pats = [('chap', chap_regex, 'm')]

        
        src_file_path = LIB.loc[book_id].source_file_path


        # Create object
        text = TextParser(src_file_path, ohco_pats=ohco_pats, 
                          clip_pats=clip_pats, use_nltk=True)


        # Define parameters
        text.verbose = True
        text.strip_hyphens = True
        text.strip_whitespace = True

        
        # Parse
        text.import_source().parse_tokens();

        

        # Name things
        text.TOKENS['book_id'] = book_id
        text.TOKENS = text.TOKENS.reset_index().set_index(['book_id'] + text.OHCO)

        # Add to list
        books.append(text.TOKENS)
        
    # Combine into a single dataframe
    CORPUS = pd.concat(books).sort_index()

    # Clean up
    del(books)
    del(text)
        
    print("Done")
        
    return CORPUS

In [74]:
CORPUS = tokenize_collection(LIB)

Tokenizing 600 NOTES FROM THE UNDERGROUND
Importing  /home/wgp3aq/Documents/MSDS/DS5001/data/gutenberg/dostoevsky-set/DOSTOEVSKY_FYODOR_NOTES_FROM_THE_UNDERGROUND-pg600.txt
Clipping text
Parsing OHCO level 0 chap_id by milestone ^(I|II|III|IV|V|VI|VII|VIII|IX|X|XI)$
line_str chap_str
Index(['chap_str'], dtype='object')
Parsing OHCO level 1 para_num by delimitter \n\n
Parsing OHCO level 2 sent_num by NLTK model
Parsing OHCO level 3 token_num by NLTK model


/home/wgp3aq/Documents/MSDS/DS5001/repo/lessons/lib/textparser.py:132: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  div_lines = self.TOKENS[src_col].str.contains(div_pat, regex=True, case=True)


Tokenizing 2197 THE GAMBLER
Importing  /home/wgp3aq/Documents/MSDS/DS5001/data/gutenberg/dostoevsky-set/DOSTOEVSKY_FYODOR_THE_GAMBLER-pg2197.txt
Clipping text
Parsing OHCO level 0 chap_id by milestone ^(I|II|III|IV|V|VI|VII|VIII|IX|X|XI|XII|XIII|XIV|XV|XVI|XVII)$
line_str chap_str
Index(['chap_str'], dtype='object')
Parsing OHCO level 1 para_num by delimitter \n\n
Parsing OHCO level 2 sent_num by NLTK model


/home/wgp3aq/Documents/MSDS/DS5001/repo/lessons/lib/textparser.py:132: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  div_lines = self.TOKENS[src_col].str.contains(div_pat, regex=True, case=True)


Parsing OHCO level 3 token_num by NLTK model
Tokenizing 2554 CRIME AND PUNISHMENT
Importing  /home/wgp3aq/Documents/MSDS/DS5001/data/gutenberg/dostoevsky-set/DOSTOEVSKY_FYODOR_CRIME_AND_PUNISHMENT-pg2554.txt
Clipping text
Parsing OHCO level 0 chap_id by milestone ^(CHAPTER|EPILOGUE)
line_str chap_str
Index(['chap_str'], dtype='object')
Parsing OHCO level 1 para_num by delimitter \n\n
Parsing OHCO level 2 sent_num by NLTK model


/home/wgp3aq/Documents/MSDS/DS5001/repo/lessons/lib/textparser.py:132: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  div_lines = self.TOKENS[src_col].str.contains(div_pat, regex=True, case=True)


Parsing OHCO level 3 token_num by NLTK model
Tokenizing 2638 THE IDIOT
Importing  /home/wgp3aq/Documents/MSDS/DS5001/data/gutenberg/dostoevsky-set/DOSTOEVSKY_FYODOR_THE_IDIOT-pg2638.txt
Clipping text
Parsing OHCO level 0 chap_id by milestone ^(I|II|III|IV|V|VI|VII|VIII|IX|X|XI|XII|XIII|XIV|XV|XVI)\.$
line_str chap_str
Index(['chap_str'], dtype='object')
Parsing OHCO level 1 para_num by delimitter \n\n
Parsing OHCO level 2 sent_num by NLTK model


/home/wgp3aq/Documents/MSDS/DS5001/repo/lessons/lib/textparser.py:132: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  div_lines = self.TOKENS[src_col].str.contains(div_pat, regex=True, case=True)


Parsing OHCO level 3 token_num by NLTK model
Tokenizing 8117 THE POSSESSED
Importing  /home/wgp3aq/Documents/MSDS/DS5001/data/gutenberg/dostoevsky-set/DOSTOEVSKY_FYODOR_THE_POSSESSED-pg8117.txt
Clipping text
Parsing OHCO level 0 chap_id by milestone ^CHAPTER.*
line_str chap_str
Index(['chap_str'], dtype='object')
Parsing OHCO level 1 para_num by delimitter \n\n
Parsing OHCO level 2 sent_num by NLTK model
Parsing OHCO level 3 token_num by NLTK model
Tokenizing 28054 THE BROTHERS KARAMAZOV
Importing  /home/wgp3aq/Documents/MSDS/DS5001/data/gutenberg/dostoevsky-set/DOSTOEVSKY_FYODOR_THE_BROTHERS_KARAMAZOV-pg28054.txt
Clipping text
Parsing OHCO level 0 chap_id by milestone ^Chapter [IVX]+\.$
line_str chap_str
Index(['chap_str'], dtype='object')
Parsing OHCO level 1 para_num by delimitter \n\n
Parsing OHCO level 2 sent_num by NLTK model
Parsing OHCO level 3 token_num by NLTK model
Done


### Book Lengths and Chapter Counts

In [75]:
CORPUS = CORPUS[CORPUS.term_str != '']

In [76]:
CORPUS['pos_group'] = CORPUS.pos.str[:2]

In [77]:
CORPUS

pos_tuple  pos  \
book_id chap_id para_num sent_num token_num                            
600     1       1        0        0                    (I, PRP)  PRP   
                                  1                   (am, VBP)  VBP   
                                  2                     (a, DT)   DT   
                                  3                  (sick, JJ)   JJ   
                                  4               (man...., NN)   NN   
...                                                         ...  ...   
28054   96      76       1        17              (“Hurrah, NN)   NN   
                                  18                  (for, IN)   IN   
                                  19         (Karamazov!”, NNP)  NNP   
                77       0        0                   (THE, DT)   DT   
                                  1                   (END, NN)   NN   

                                               token_str   term_str pos_group  
book_id chap_id para_num sent_num token_num                                    
600     1       1        0        0                    I          i        PR  
                                  1                   am         am        VB  
                                  2                    a          a        DT  
                                  3                 sick       sick        JJ  
                                  4              man....        man        NN  
...                                                  ...        ...       ...  
28054   96      76       1        17             “Hurrah     hurrah        NN  
                                  18                 for        for        IN  
                                  19         Karamazov!”  karamazov        NN  
                77       0        0                  THE        the        DT  
                                  1                  END        end        NN  

[1155871 rows x 5 columns]

## Finishing Off LIB Table by adding book_len and n_chaps

In [79]:
LIB['book_len'] = CORPUS.groupby('book_id').term_str.count()
LIB['n_chaps'] = CORPUS.reset_index()[['book_id','chap_id']]\
    .drop_duplicates()\
    .groupby('book_id').chap_id.count()
LIB

,source_file_path,author,title,date,type,translator,chap_regex,book_len,n_chaps
book_id,,,,,,,,,
600,/home/wgp3aq/Documents/MSDS/DS5001/data/gutenb...,"DOSTOEVSKY, FYODOR",NOTES FROM THE UNDERGROUND,1864,novella,Constance Garnett,^(I|II|III|IV|V|VI|VII|VIII|IX|X|XI)$,43981,21
2197,/home/wgp3aq/Documents/MSDS/DS5001/data/gutenb...,"DOSTOEVSKY, FYODOR",THE GAMBLER,1866,novella,C. J. Hogarth,^(I|II|III|IV|V|VI|VII|VIII|IX|X|XI|XII|XIII|X...,60502,17
2554,/home/wgp3aq/Documents/MSDS/DS5001/data/gutenb...,"DOSTOEVSKY, FYODOR",CRIME AND PUNISHMENT,1866,novel,Constance Garnett,^(CHAPTER|EPILOGUE),204177,40
2638,/home/wgp3aq/Documents/MSDS/DS5001/data/gutenb...,"DOSTOEVSKY, FYODOR",THE IDIOT,1869,novel,Eva Martin,^(I|II|III|IV|V|VI|VII|VIII|IX|X|XI|XII|XIII|X...,242278,50
8117,/home/wgp3aq/Documents/MSDS/DS5001/data/gutenb...,"DOSTOEVSKY, FYODOR",THE POSSESSED,1872,novel,Constance Garnett,^CHAPTER.*,255013,23
28054,/home/wgp3aq/Documents/MSDS/DS5001/data/gutenb...,"DOSTOEVSKY, FYODOR",THE BROTHERS KARAMAZOV,1880,novel,Constance Garnett,^Chapter [IVX]+\.$,349220,96


## Creating VOCAB Table

**This is the basic VOCAB table, final version (with'dfidf' column) is completed later, and top 20 words are determined later, after the creation of the dfidf table**

In [80]:
VOCAB = CORPUS.term_str.value_counts().to_frame('n').sort_index()
VOCAB.index.name = 'term_str'
VOCAB['n_chars'] = VOCAB.index.str.len()
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()
VOCAB['i'] = -np.log2(VOCAB.p)

**Max POS**

In [81]:
VOCAB['max_pos'] = CORPUS[['term_str','pos']].value_counts().unstack(fill_value=0).idxmax(1)

In [82]:
VOCAB['max_pos_group'] = CORPUS[['term_str','pos_group']].value_counts().unstack(fill_value=0).idxmax(1)

**Stopwords**

In [83]:
sw = pd.DataFrame(nltk.corpus.stopwords.words('english'), columns=['term_str'])
sw = sw.reset_index().set_index('term_str')
sw.columns = ['dummy']
sw.dummy = 1

In [84]:
VOCAB['stop'] = VOCAB.index.map(sw.dummy)
VOCAB['stop'] = VOCAB['stop'].fillna(0).astype('int')

**Stems**

In [85]:
from nltk.stem.porter import PorterStemmer
stemmer1 = PorterStemmer()
VOCAB['porter_stem'] = VOCAB.apply(lambda x: stemmer1.stem(x.name), 1)

In [86]:
VOCAB

,n,n_chars,p,i,max_pos,max_pos_group,stop,porter_stem
term_str,,,,,,,,
1,2,1,1.731345e-06,19.139675,CD,CD,0,1
100,5,3,4.328364e-06,17.817747,CD,CD,0,100
108000,1,6,8.656727e-07,20.139675,CD,CD,0,108000
11th,1,4,8.656727e-07,20.139675,CD,CD,0,11th
120,3,3,2.597018e-06,18.554712,CD,CD,0,120
...,...,...,...,...,...,...,...,...
étoiles,3,7,2.597018e-06,18.554712,IN,IN,0,étoil
éveillé,1,7,8.656727e-07,20.139675,NN,NN,0,éveillé
êtes,6,4,5.194036e-06,17.554712,NNS,NN,0,ête


# Derived Tables

**Function to get BOW tables**

In [87]:
def gen_BOW(tokens_df, bag_lvl):
    
    OHCO = ['book_id', 'chap_id', 'para_num', 'sent_num', 'token_num']
    
    bags = dict(
        SENTS = OHCO[:4],
        PARAS = OHCO[:3],
        CHAPS = OHCO[:2],
        BOOKS = OHCO[:1]
    )
    
    BOW = tokens_df.groupby(
        bags[bag_lvl]+['term_str']).term_str.count().to_frame('n')
    return BOW

**Function to get DTM, DFIDF, TFIDF from BOW**

In [88]:
def gen_TFIDF(BOW, TF_type):
    ##### Document-Term count matrix
    DTCM = BOW.n.unstack(fill_value=0)
    
    # define parameters
    tf_method = TF_type     
    tf_norm_k = .5          
    idf_method = 'standard'  
    gradient_cmap = 'YlGnBu' 
    
    ##### get TF
    tf = {
        'sum': (DTCM.T / DTCM.T.sum()).T,
        'max': (DTCM.T / DTCM.T.max()).T,
        'log': (np.log2(1 + DTCM.T)).T,
        'raw':  DTCM,
        'double_norm': (DTCM.T / DTCM.T.max()).T,
        'binary': DTCM.T.astype('bool').astype('int').T
    }    
    TF = tf[tf_method]
    
    ##### get DF
    DF = DTCM.astype('bool').sum()
    
    ##### get IDF
    N = DTCM.shape[0]
    IDF = np.log2(N / DF)
    
    ##### get/return TFIDF
    TFIDF = TF * IDF
    
    ##### get DFIDF
    length = BOW.index.droplevel(-1).nunique()
    DFIDF = DF * np.log2(length/DF)  
    
    ##### get DTM
    DTM = DTCM
    
    return DTM, DFIDF, TFIDF
    
    

### Creating BOW Table

**Apply these functions to CORPUS with chapters as bags, 'max' as TF count method**

In [89]:
BOW = gen_BOW(CORPUS, 'CHAPS')
BOW

n
book_id chap_id term_str      
600     1       a           31
                about        4
                absolutely   2
                active       1
                acutest      1
...                         ..
28054   96      you         37
                youll        2
                your         9
                yours        2
                youve        1

[276464 rows x 1 columns]

### Creating DTM, DFIDF, TFIDF Tables

In [90]:
DTM, DFIDF, TFIDF = gen_TFIDF(BOW, 'max')
DTM

term_str         1  100  108000  11th  120  1200  13  14  1413  1428  ...  \
book_id chap_id                                                       ...   
600     1        0    0       0     0    0     0   0   0     0     0  ...   
        2        0    0       0     0    0     0   0   0     0     0  ...   
        3        0    0       0     0    0     0   0   0     0     0  ...   
        4        0    0       0     0    0     0   0   0     0     0  ...   
        5        0    0       0     0    0     0   0   0     0     0  ...   
...             ..  ...     ...   ...  ...   ...  ..  ..   ...   ...  ...   
28054   92       0    0       0     0    0     0   0   0     0     0  ...   
        93       0    0       0     0    0     0   0   0     0     0  ...   
        94       0    0       0     0    0     0   0   0     0     0  ...   
        95       0    0       0     0    0     0   0   0     0     0  ...   
        96       0    0       0     0    0     0   0   0     0     0  ...   

term_str         étage  étaient  étais  étape  éternelle  étoiles  éveillé  \
book_id chap_id                                                              
600     1            0        0      0      0          0        0        0   
        2            0        0      0      0          0        0        0   
        3            0        0      0      0          0        0        0   
        4            0        0      0      0          0        0        0   
        5            0        0      0      0          0        0        0   
...                ...      ...    ...    ...        ...      ...      ...   
28054   92           0        0      0      0          0        0        0   
        93           0        0      0      0          0        0        0   
        94           0        0      0      2          0        0        0   
        95           0        0      0      1          0        0        0   
        96           0        0      0      0          0        0        0   

term_str         êtes  être  œcumenical  
book_id chap_id                          
600     1           0     0           0  
        2           0     0           0  
        3           0     0           0  
        4           0     0           0  
        5           0     0           0  
...               ...   ...         ...  
28054   92          0     0           0  
        93          0     0           0  
        94          0     0           0  
        95          0     0           0  
        96          0     0           0  

[247 rows x 24867 columns]

In [91]:
TFIDF

term_str           1  100  108000  11th  120  1200   13   14  1413  1428  ...  \
book_id chap_id                                                           ...   
600     1        0.0  0.0     0.0   0.0  0.0   0.0  0.0  0.0   0.0   0.0  ...   
        2        0.0  0.0     0.0   0.0  0.0   0.0  0.0  0.0   0.0   0.0  ...   
        3        0.0  0.0     0.0   0.0  0.0   0.0  0.0  0.0   0.0   0.0  ...   
        4        0.0  0.0     0.0   0.0  0.0   0.0  0.0  0.0   0.0   0.0  ...   
        5        0.0  0.0     0.0   0.0  0.0   0.0  0.0  0.0   0.0   0.0  ...   
...              ...  ...     ...   ...  ...   ...  ...  ...   ...   ...  ...   
28054   92       0.0  0.0     0.0   0.0  0.0   0.0  0.0  0.0   0.0   0.0  ...   
        93       0.0  0.0     0.0   0.0  0.0   0.0  0.0  0.0   0.0   0.0  ...   
        94       0.0  0.0     0.0   0.0  0.0   0.0  0.0  0.0   0.0   0.0  ...   
        95       0.0  0.0     0.0   0.0  0.0   0.0  0.0  0.0   0.0   0.0  ...   
        96       0.0  0.0     0.0   0.0  0.0   0.0  0.0  0.0   0.0   0.0  ...   

term_str         étage  étaient  étais     étape  éternelle  étoiles  éveillé  \
book_id chap_id                                                                 
600     1          0.0      0.0    0.0  0.000000        0.0      0.0      0.0   
        2          0.0      0.0    0.0  0.000000        0.0      0.0      0.0   
        3          0.0      0.0    0.0  0.000000        0.0      0.0      0.0   
        4          0.0      0.0    0.0  0.000000        0.0      0.0      0.0   
        5          0.0      0.0    0.0  0.000000        0.0      0.0      0.0   
...                ...      ...    ...       ...        ...      ...      ...   
28054   92         0.0      0.0    0.0  0.000000        0.0      0.0      0.0   
        93         0.0      0.0    0.0  0.000000        0.0      0.0      0.0   
        94         0.0      0.0    0.0  0.182852        0.0      0.0      0.0   
        95         0.0      0.0    0.0  0.073141        0.0      0.0      0.0   
        96         0.0      0.0    0.0  0.000000        0.0      0.0      0.0   

term_str         êtes  être  œcumenical  
book_id chap_id                          
600     1         0.0   0.0         0.0  
        2         0.0   0.0         0.0  
        3         0.0   0.0         0.0  
        4         0.0   0.0         0.0  
        5         0.0   0.0         0.0  
...               ...   ...         ...  
28054   92        0.0   0.0         0.0  
        93        0.0   0.0         0.0  
        94        0.0   0.0         0.0  
        95        0.0   0.0         0.0  
        96        0.0   0.0         0.0  

[247 rows x 24867 columns]

### Joining TFIDF, DFIDF into BOW 

In [92]:
# organize
TFIDF_stacked = TFIDF.stack().rename('tfidf').to_frame()

# mean tfidf calculations
BOW_tfidf= TFIDF_stacked.groupby(['term_str']).agg(
    {'tfidf':'mean'}).sort_values('tfidf', ascending=False)
BOW_tfidf = BOW_tfidf.rename(columns={'tfidf': 'tfidf'})
BOW_tfidf

BOW = BOW.join(BOW_tfidf, how='left')

DFIDF_df = DFIDF.rename('dfidf').to_frame()
BOW = BOW.join(DFIDF_df, how='left')

BOW

n     tfidf       dfidf
book_id chap_id term_str                            
600     1       a           31  0.000000    0.000000
                about        4  0.001396    5.723799
                absolutely   2  0.004361  131.092106
                active       1  0.002208   74.677046
                acutest      1  0.000593   19.090214
...                         ..       ...         ...
28054   96      you         37  0.010036    5.723799
                youll        2  0.006364  126.830876
                your         9  0.007925   23.661752
                yours        2  0.004557  131.082085
                youve        1  0.007494  119.857694

[276464 rows x 3 columns]

### Joining DFIDF into VOCAB 

In [93]:
VOCAB = VOCAB.join(DFIDF_df, how='left')
VOCAB

,n,n_chars,p,i,max_pos,max_pos_group,stop,porter_stem,dfidf
term_str,,,,,,,,,
1,2,1,1.731345e-06,19.139675,CD,CD,0,1,13.896734
100,5,3,4.328364e-06,17.817747,CD,CD,0,100,19.090214
108000,1,6,8.656727e-07,20.139675,CD,CD,0,108000,7.948367
11th,1,4,8.656727e-07,20.139675,CD,CD,0,11th,7.948367
120,3,3,2.597018e-06,18.554712,CD,CD,0,120,19.090214
...,...,...,...,...,...,...,...,...,...
étoiles,3,7,2.597018e-06,18.554712,IN,IN,0,étoil,13.896734
éveillé,1,7,8.656727e-07,20.139675,NN,NN,0,éveillé,7.948367
êtes,6,4,5.194036e-06,17.554712,NNS,NN,0,ête,19.090214


### Top 20 significant words in the corpus by DFIDF (Part of VOCAB Table Requirements):

In [94]:
# top 20 significant words in the corpus by DFIDF:
VOCAB_dfidf_top20 = VOCAB.sort_values(by='dfidf',ascending=False)[:20]
VOCAB_dfidf_top20

,n,n_chars,p,i,max_pos,max_pos_group,stop,porter_stem,dfidf
term_str,,,,,,,,,
frightened,183,10,0.000158,12.623975,VBN,VB,0,frighten,131.092106
forget,140,6,0.000121,13.010392,VB,VB,0,forget,131.092106
absolutely,151,10,0.000131,12.901270,RB,RB,0,absolut,131.092106
further,134,7,0.000116,13.073586,JJ,JJ,1,further,131.092106
grew,126,4,0.000109,13.162395,VBD,VB,0,grew,131.092106
curiosity,154,9,0.000133,12.872888,NN,NN,0,curios,131.092106
die,189,3,0.000164,12.577433,VB,VB,0,die,131.086272
directly,159,8,0.000138,12.826792,NN,NN,0,directli,131.086272
giving,136,6,0.000118,13.052212,VBG,VB,0,give,131.086272


### Creating Reduced and Normalized TFIDF_L2

In [95]:
# choose top 1000 term_str, ordered by dfidf
top_1000 = BOW.sort_values(
    by="dfidf", ascending=False).reset_index()
top_1000 = top_1000.drop_duplicates(
    subset="term_str").head(1000).set_index(["book_id", "chap_id", "term_str"])

In [96]:
# make the TFIDF matrix with top 1000 most significant terms, with dfidf as sig
top_1000_term_str = top_1000.index.get_level_values("term_str")
TFIDF_1000 = TFIDF.loc[:, TFIDF.columns.isin(
    top_1000_term_str)]

In [97]:
# L2 TFIDF table with top 1000 significance terms, grouping by book and chapter to use in PCA later
TFIDF_means = TFIDF_1000.groupby(['book_id', 'chap_id']).mean()
L2 = TFIDF_means.apply(lambda x: x / norm(x), 1) # Pythagorean, AKA Euclidean
L2

term_str             able     above    abroad  absolute  absolutely    absurd  \
book_id chap_id                                                                 
600     1        0.000000  0.000000  0.000000       0.0    0.111357  0.000000   
        2        0.086255  0.000000  0.000000       0.0    0.045269  0.052258   
        3        0.000000  0.024283  0.000000       0.0    0.032451  0.000000   
        4        0.000000  0.000000  0.000000       0.0    0.079448  0.000000   
        5        0.039475  0.000000  0.000000       0.0    0.000000  0.000000   
...                   ...       ...       ...       ...         ...       ...   
28054   92       0.016162  0.019042  0.000000       0.0    0.000000  0.000000   
        93       0.000000  0.000000  0.000000       0.0    0.000000  0.040290   
        94       0.000000  0.000000  0.070957       0.0    0.000000  0.000000   
        95       0.009151  0.000000  0.000000       0.0    0.000000  0.000000   
        96       0.000000  0.013438  0.000000       0.0    0.000000  0.000000   

term_str           accept  according   account  acquaintance  ...     wrote  \
book_id chap_id                                               ...             
600     1        0.000000   0.000000  0.000000      0.000000  ...  0.070308   
        2        0.000000   0.000000  0.000000      0.000000  ...  0.000000   
        3        0.121548   0.000000  0.021875      0.000000  ...  0.000000   
        4        0.000000   0.000000  0.000000      0.000000  ...  0.000000   
        5        0.000000   0.000000  0.000000      0.000000  ...  0.000000   
...                   ...        ...       ...           ...  ...       ...   
28054   92       0.031771   0.136086  0.000000      0.000000  ...  0.000000   
        93       0.000000   0.000000  0.000000      0.043088  ...  0.000000   
        94       0.000000   0.000000  0.015707      0.000000  ...  0.000000   
        95       0.000000   0.000000  0.000000      0.000000  ...  0.000000   
        96       0.000000   0.000000  0.000000      0.000000  ...  0.000000   

term_str             yard     year  yesterday  youd     youll  youre  \
book_id chap_id                                                        
600     1        0.000000  0.05042   0.000000   0.0  0.000000    0.0   
        2        0.000000  0.00000   0.000000   0.0  0.000000    0.0   
        3        0.000000  0.00000   0.000000   0.0  0.000000    0.0   
        4        0.000000  0.00000   0.000000   0.0  0.000000    0.0   
        5        0.000000  0.00000   0.000000   0.0  0.000000    0.0   
...                   ...      ...        ...   ...       ...    ...   
28054   92       0.000000  0.00000   0.000000   0.0  0.000000    0.0   
        93       0.000000  0.00000   0.000000   0.0  0.000000    0.0   
        94       0.000000  0.00000   0.000000   0.0  0.000000    0.0   
        95       0.023049  0.00000   0.022313   0.0  0.000000    0.0   
        96       0.000000  0.00000   0.000000   0.0  0.027496    0.0   

term_str            yours     youth     youve  
book_id chap_id                                
600     1        0.000000  0.075305  0.000000  
        2        0.000000  0.000000  0.000000  
        3        0.000000  0.000000  0.000000  
        4        0.000000  0.000000  0.000000  
        5        0.000000  0.000000  0.000000  
...                   ...       ...       ...  
28054   92       0.000000  0.034417  0.000000  
        93       0.000000  0.000000  0.000000  
        94       0.000000  0.000000  0.000000  
        95       0.028501  0.000000  0.009151  
        96       0.035522  0.000000  0.011405  

[247 rows x 1000 columns]

# Save Ouputs

In [98]:
LIB.to_csv(f"{output_dir}/{data_prefix}-LIB.csv", index=True)
CORPUS.to_csv(f"{output_dir}/{data_prefix}-CORPUS.csv", index=True)
VOCAB.to_csv(f"{output_dir}/{data_prefix}-VOCAB.csv", index=True)
BOW.to_csv(f"{output_dir}/{data_prefix}-BOW.csv", index=True)
DTM.to_csv(f"{output_dir}/{data_prefix}-DTM.csv", index=True)
TFIDF.to_csv(f"{output_dir}/{data_prefix}-TFIDF.csv", index=True)
L2.to_csv(f"{output_dir}/{data_prefix}-TFIDF_L2.csv", index=True)